# This file is for Task 2 - Exploratory Data Analysis (EDA) of the project.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer


# local imports
import utils

## Load the dataset

In [ ]:
train_df, dev_df, test_df = utils.load_and_clean_data()

In [ ]:
def split_stats(train_df, dev_df, test_df):
    import pandas as pd
    import numpy as np

    dfs = {"Train": train_df, "Dev": dev_df, "Test": test_df}

    # ── Cross-split duplicate check ─────────────────────────────────────────
    all_texts = pd.concat([df["text"] for df in dfs.values()], ignore_index=True)
    n_dupes = all_texts.duplicated().sum()
    print(f"Duplicate texts across all splits: {n_dupes}\n")

    # ── Class distribution table ─────────────────────────────────────────────
    class_rows = []
    for split, df in dfs.items():
        if "binary_label" not in df.columns:
            class_rows.append({"Split": split, "Total": len(df), "No PCL (0)": "-",
                                "PCL (1)": "-", "% PCL": "-", "Ratio (0:1)": "-"})
            continue
        counts = df["binary_label"].value_counts().sort_index()
        total  = len(df)
        class_rows.append({
            "Split":       split,
            "Total":       total,
            "No PCL (0)":  counts[0],
            "PCL (1)":     counts[1],
            "% PCL":       f"{counts[1]/total*100:.1f}%",
            "Ratio (0:1)": f"{counts[0]/counts[1]:.1f}",
        })

    # ── Word count stats table ───────────────────────────────────────────────
    len_rows = []
    for split, df in dfs.items():
        df = df.copy()
        df["wc"] = df["text"].str.split().str.len()

        groups = {"Overall": df}
        if "binary_label" in df.columns:
            groups["No PCL"] = df[df["binary_label"] == 0]
            groups["PCL"]    = df[df["binary_label"] == 1]

        for grp_name, grp in groups.items():
            wc = grp["wc"]
            len_rows.append({
                "Split":   split,
                "Class":   grp_name,
                "N":       len(wc),
                "Min":     int(wc.min()),
                "Mean":    f"{wc.mean():.1f}",
                "Median":  int(wc.median()),
                "Max":     int(wc.max()),
                "Std":     f"{wc.std():.1f}",
                "p90":     int(np.percentile(wc, 90)),
                "p95":     int(np.percentile(wc, 95)),
                "p99":     int(np.percentile(wc, 99)),
            })

    return pd.DataFrame(class_rows), pd.DataFrame(len_rows)


class_rows, len_rows = split_stats(train_df, dev_df, test_df)

In [ ]:
print("Class Distribution")
class_rows

In [ ]:
print("Word Count Statistics")
len_rows

In [ ]:
def plot_token_count_histograms(train_df, dev_df, test_df):
    train_df = train_df.copy()
    dev_df   = dev_df.copy()
    test_df  = test_df.copy()

    train_df["word_count"] = train_df["text"].str.split().str.len()
    dev_df["word_count"]   = dev_df["text"].str.split().str.len()
    test_df["word_count"]  = test_df["text"].str.split().str.len()

    train_pcl    = train_df[train_df["binary_label"] == 1]["word_count"].values
    train_no_pcl = train_df[train_df["binary_label"] == 0]["word_count"].values
    dev_pcl      = dev_df[dev_df["binary_label"] == 1]["word_count"].values
    dev_no_pcl   = dev_df[dev_df["binary_label"] == 0]["word_count"].values

    bins = np.linspace(0, 300, 30)

    def hist_vals(data, bins):
        counts, edges = np.histogram(data, bins=bins, density=True)
        x = (edges[:-1] + edges[1:]) / 2
        return x, counts

    c_pcl,    h_pcl    = hist_vals(np.concatenate([train_pcl, dev_pcl]), bins)
    c_no_pcl, h_no_pcl = hist_vals(np.concatenate([train_no_pcl, dev_no_pcl]), bins)
    c_train_all,    h_train_all    = hist_vals(np.concatenate([train_pcl, train_no_pcl]), bins)
    c_dev_all,      h_dev_all      = hist_vals(np.concatenate([dev_pcl,   dev_no_pcl]),   bins)
    c_test_all,     h_test_all     = hist_vals(test_df["word_count"].values, bins)

    fig = make_subplots(rows=1, cols=2, subplot_titles=[
        "PCL vs No-PCL word count (train+dev)",
        "Train vs Dev vs Test word count"
    ])

    # Panel 1
    fig.add_trace(go.Scatter(x=c_no_pcl, y=h_no_pcl, name="No PCL",
                             mode="none", fill="tozeroy",
                             fillcolor="rgba(99,110,250,0.35)",
                             legendgroup="panel1", legendgrouptitle_text="Class"),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=c_pcl,    y=h_pcl,    name="PCL",
                             mode="none", fill="tozeroy",
                             fillcolor="rgba(239,85,59,0.35)",
                             legendgroup="panel1"),
                  row=1, col=1)

    # Panel 2
    fig.add_trace(go.Scatter(x=c_train_all, y=h_train_all, name="Train",
                             mode="none", fill="tozeroy",
                             fillcolor="rgba(99,110,250,0.35)",
                             legendgroup="panel2", legendgrouptitle_text="Split"),
                  row=1, col=2)
    fig.add_trace(go.Scatter(x=c_dev_all,   y=h_dev_all,   name="Dev",
                             mode="none", fill="tozeroy",
                             fillcolor="rgba(0,204,150,0.35)",
                             legendgroup="panel2"),
                  row=1, col=2)
    fig.add_trace(go.Scatter(x=c_test_all,  y=h_test_all,  name="Test",
                             mode="none", fill="tozeroy",
                             fillcolor="rgba(239,85,59,0.35)",
                             legendgroup="panel2"),
                  row=1, col=2)


    fig.update_layout(
        barmode="overlay",
        legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
    )
    fig.update_xaxes(title_text="Word count")
    fig.update_yaxes(title_text="Density", row=1, col=1)
    fig.update_yaxes(title_text="Density", row=1, col=2)

    fig.show()


plot_token_count_histograms(train_df, dev_df, test_df)


In [ ]:
pcl_texts    = train_df[train_df["binary_label"] == 1]["text"].tolist()
no_pcl_texts = train_df[train_df["binary_label"] == 0]["text"].tolist()

# ══════════════════════════════════════════════════════════════════
# CHART 1 — TF-IDF distinctive 2–3-grams per class
# ══════════════════════════════════════════════════════════════════
vec = TfidfVectorizer(ngram_range=(2, 3), stop_words="english", max_features=8000)
X = vec.fit_transform(train_df["text"])
terms = vec.get_feature_names_out()

pcl_mask    = (train_df["binary_label"] == 1).values
no_pcl_mask = (train_df["binary_label"] == 0).values

pcl_mean    = X[pcl_mask].mean(axis=0).A1
no_pcl_mean = X[no_pcl_mask].mean(axis=0).A1

# Distinctiveness = high in one class, low in the other
pcl_distinctive    = pcl_mean - no_pcl_mean
no_pcl_distinctive = no_pcl_mean - pcl_mean

N = 15
top_pcl    = sorted(zip(terms, pcl_distinctive),    key=lambda x: -x[1])[:N]
top_no_pcl = sorted(zip(terms, no_pcl_distinctive), key=lambda x: -x[1])[:N]

pcl_t, pcl_s       = zip(*reversed(top_pcl))
no_pcl_t, no_pcl_s = zip(*reversed(top_no_pcl))
fig1 = make_subplots(rows=1, cols=2, subplot_titles=["PCL — top phrases", "No PCL — top phrases"])

fig1.add_trace(go.Bar(x=list(pcl_s), y=list(pcl_t), orientation="h",
                      name="PCL", marker_opacity=0.85), row=1, col=1)
fig1.add_trace(go.Bar(x=list(no_pcl_s), y=list(no_pcl_t), orientation="h",
                      name="No PCL", marker_opacity=0.85), row=1, col=2)

fig1.update_layout(
    title={"text": "Most Distinctive 2-3-grams by Class (TF-IDF)<br>"},
    showlegend=False,
)
fig1.update_xaxes(title_text="TF-IDF score")
fig1.update_traces(cliponaxis=False)
fig1.show()


In [ ]:
# ══════════════════════════════════════════════════════════════════
# CHART 2 — Log-odds diverging bar chart
# ══════════════════════════════════════════════════════════════════
def get_ngram_counts(texts, ngram_range=(2, 3)):
    vec = CountVectorizer(ngram_range=ngram_range, stop_words="english", min_df=3)  # was min_df=2
    X = vec.fit_transform(texts)
    return dict(zip(vec.get_feature_names_out(), X.sum(axis=0).A1))

pcl_counts    = get_ngram_counts(pcl_texts)
no_pcl_counts = get_ngram_counts(no_pcl_texts)

all_terms = set(pcl_counts) | set(no_pcl_counts)

log_odds = {
    t: np.log((pcl_counts.get(t, 0) + 1) / (no_pcl_counts.get(t, 0) + 1))
    for t in all_terms
}

N = 20
sorted_lo     = sorted(log_odds.items(), key=lambda x: x[1])
top_no_pcl_lo = sorted_lo[:N]
top_pcl_lo    = sorted_lo[-N:][::-1]

combined  = top_no_pcl_lo + top_pcl_lo
terms_lo  = [t for t, _ in combined]
scores_lo = [s for _, s in combined]
colors    = ["#ef553b" if s > 0 else "#636efa" for s in scores_lo]

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    x=scores_lo, y=terms_lo, orientation="h",
    marker_color=colors, marker_opacity=0.85,
))
fig2.add_vline(x=0, line_width=1.5, line_color="white")
# Calculate height dynamically based on number of bars
n_bars = len(terms_lo)
bar_height = 20          # px per bar
fig_height = bar_height * n_bars + 150   # +150 for title/margins

fig2.update_layout(
    height=fig_height,
    margin=dict(l=220, r=40, t=90, b=50),   # wide left margin for n-gram labels
    yaxis=dict(
        categoryorder="array",
        categoryarray=terms_lo,
        tickmode="array",           # force ALL ticks to be shown
        tickvals=terms_lo,          # explicitly list every term
        ticktext=terms_lo,
        automargin=True,            # auto-expand margin if labels still overflow
    ),
)

fig2.update_layout(
    title={"text": "Log-Odds Ratio: PCL vs No-PCL distinctive terms<br>"
                   "<span style='font-size:15px;font-weight:normal'>"
                   "Red = PCL-leaning  |  Blue = No-PCL-leaning</span>"},
    yaxis=dict(categoryorder="array", categoryarray=terms_lo),
)
fig2.update_xaxes(title_text="Log-odds (PCL)")
fig2.update_traces(cliponaxis=False)
fig2.show()